# 오디오 기반 RAG


In [42]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

AZURE_TRANSCRIBE_API_KEY = os.getenv('AZURE_TRANSCRIBE_API_KEY')
AZURE_TRANSCRIBE_ENDPOINT = os.getenv('AZURE_TRANSCRIBE_ENDPOINT')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'RAG'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    if len(os.getenv('LANGCHAIN_API_KEY')) > 0:
        print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

43b13g4OZS
gpt-5-mini


---
## 1. ASR(automatic speech recognition)을 사용해 Text로 변환하여 저장
- 폴더의 오디오 파일 목록을 읽고, 간단한 메타데이터(길이 등)를 수집합니다.

오디오 파일 ref : https://blog.naver.com/adsound_rec/221835489818

In [43]:
%pip install soundfile

Note: you may need to restart the kernel to use updated packages.


In [44]:
from typing import List, Dict
import soundfile as sf
import os, sys, json, time
from pathlib import Path

AUDIO_DIR = Path("audio")

SUPPORTED = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

# ----------------------------
# 1. 오디오 파일 스캔
# ----------------------------
def scan_audio_files(folder: Path) -> List[Dict]:
    items = []
    for p in folder.rglob("*"):
        if p.suffix.lower() in SUPPORTED:
            try:
                f = sf.SoundFile(str(p))
                duration = len(f) / f.samplerate
                items.append({
                    "path": str(p),
                    "duration": duration,
                    "samplerate": f.samplerate
                })
            except Exception as e:
                items.append({"path": str(p), "error": str(e)})
    return items

files = scan_audio_files(AUDIO_DIR)
print("Found", len(files), "files")

Found 3 files


In [45]:
files

[{'path': 'audio/샘플_1.wav',
  'duration': 25.39204081632653,
  'samplerate': 44100},
 {'path': 'audio/샘플_2.wav',
  'duration': 30.01689342403628,
  'samplerate': 44100},
 {'path': 'audio/샘플_3.wav',
  'duration': 23.682879818594106,
  'samplerate': 44100}]

In [46]:
# audio파일을 ipynb에서 재생
from IPython.display import Audio, display
display(Audio(files[0]['path'])) 

## ASR

open ai에서 최근에 제공한 audio to text 모델인 gpt-4o-mini-transcribe 을 사용해 text로 변환합니다.

https://platform.openai.com/docs/models/gpt-4o-mini-transcribe

In [47]:
from openai import AzureOpenAI
client = AzureOpenAI(
    api_key=AZURE_TRANSCRIBE_API_KEY,
    azure_endpoint=AZURE_TRANSCRIBE_ENDPOINT,
    api_version="2024-12-01-preview",
)

def transcribe_file(path: str) -> Dict:

    with open(path, "rb") as f:
        result = client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",   
            file=f,
            response_format="json",        
        )
    return {"path": path, "text": result.text}

transcripts: List[Dict] = []
for item in files:
    if "error" in item: 
        continue
    transcripts.append(transcribe_file(item["path"]))

print("Transcribed:", len(transcripts))

Transcribed: 3


In [48]:
transcripts

[{'path': 'audio/샘플_1.wav',
  'text': '코로나19 예방수칙입니다. 손을 자주 씻기, 마스크 착용하기, 기침할 땐 입과 코 가리기, 발열, 기침, 인후통 등 증상 의심 시에는 1339 또는 보건소와 상담하시기 바랍니다.'},
 {'path': 'audio/샘플_2.wav',
  'text': '안녕하세요. 마스크 착용, 손 씻기 등 예방수칙을 꼭 지켜주세요. 코로나19 증상 의심 시 보건소 또는 질병관리본부 콜센터 1399로 연락주세요. 코로나19 함께하면 이겨낼 수 있습니다.'},
 {'path': 'audio/샘플_3.wav',
  'text': '기침 등 호흡기 증상이 있을 경우 마스크 착용하기, 의료기관 방문 시 의료진에게 해외여행력 알리기, 중국 방문 후 발열, 호흡기 증상 발생 시 보건소와 상담하세요.'}]

In [49]:
print(transcripts[0])

{'path': 'audio/샘플_1.wav', 'text': '코로나19 예방수칙입니다. 손을 자주 씻기, 마스크 착용하기, 기침할 땐 입과 코 가리기, 발열, 기침, 인후통 등 증상 의심 시에는 1339 또는 보건소와 상담하시기 바랍니다.'}


### 문서화 & 분할

text 파일을 메타데이터와 함께 Document 타입으로 변경하여 list로 저장.

Doc의 list가 일반적인 데이터 로더의 결과입니다.

In [50]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
docs = []
for t in transcripts:
    chunks = splitter.split_text(t["text"])
    for i, c in enumerate(chunks):
        docs.append(Document(
            page_content=c,
            metadata={"source": t["path"], "chunk": i}
        ))

print("Doc chunks:", len(docs))

Doc chunks: 3


### DB 저장

분할된 데이터를 임베딩한 후 외부 경로에 저장합니다.

In [51]:
from langchain_openai import AzureOpenAIEmbeddings
emb = AzureOpenAIEmbeddings(
    model=EMB_MODEL_NAME,                      
    api_key=AZURE_OPENAI_EMB_API_KEY,
    azure_endpoint=EMB_END_POINT,
    api_version="2024-08-01-preview"
)

db = Chroma.from_documents(
    docs, emb,
    # persist_directory="chroma_audio_index"
)

retriever = db.as_retriever(search_kwargs={"k": 3})

### 오디오 검색 테스트

text에 대해서 제대로 원문을 찾는지 확인

In [52]:
def search_audio(query: str):
    results = retriever.get_relevant_documents(query)
    out = []
    for r in results:
        out.append({
            "sim": getattr(r, "score", None),
            "preview": r.page_content[:100],
            "source": r.metadata["source"]
        })
    return out

print(search_audio("코로나 증상이 의심되면 어디에 연락해야해?"))

[{'sim': None, 'preview': '코로나19 예방수칙입니다. 손을 자주 씻기, 마스크 착용하기, 기침할 땐 입과 코 가리기, 발열, 기침, 인후통 등 증상 의심 시에는 1339 또는 보건소와 상담하시기 바랍니다.', 'source': 'audio/샘플_1.wav'}, {'sim': None, 'preview': '코로나19 예방수칙입니다. 손을 자주 씻기, 마스크 착용하기, 기침할 땐 입과 코 가리기, 발열, 기침, 인후통 등 증상 의심 시에는 1339 또는 보건소와 상담하시기 바랍니다.', 'source': 'audio/샘플_1.wav'}, {'sim': None, 'preview': '안녕하세요. 마스크 착용, 손 씻기 등 예방수칙을 꼭 지켜주세요. 코로나19 증상 의심 시 보건소 또는 질병관리본부 콜센터 1399로 연락주세요. 코로나19 함께하면 이겨낼 수 있', 'source': 'audio/샘플_2.wav'}]



##  LangChain 결합

이제 랭체인과 결합해 간단한 RAG 모델을 만들어봅시다.

In [53]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version="2024-12-01-preview", 
)

prompt = ChatPromptTemplate.from_template(
    """
    너는 RAG 검색 비서다. 사용자의 질문에 대해 검색 결과를 바탕으로 대답해.
    최대한 컨텍스트를 활용하되, 컨텍스트에 없는 내용은 추론하지 말고 
    "해당 질문에 대한 내용을 오디오에서 찾을 수 없습니다"고 답해.
    질문: {question}
    컨텍스트:
    {context}
    한국어로 간결히 답하고, 파일 경로를 함께 제시해.
    파일 경로는 대답의 마지막에 [참고파일] file_name 형태로 표시해.
    찾을 수 없었다면 마지막에 [참고파일 없음] 이라고 표시해.
    """
)
def format_docs(docs):
    return "\n\n---\n\n" + "\n\n---\n\n".join(d.page_content + f"\n(source: {d.metadata.get('source')})" for d in docs)

chain = (
    {"question": RunnablePassthrough(),
        "context": (retriever | format_docs)}
    | prompt | llm | StrOutputParser()
)

In [54]:
print(chain.invoke("코로나 증상이 의심되면 어디에 연락해야해?"))

코로나 증상이 의심되면 1339 또는 보건소에 연락하거나, 질병관리본부 콜센터 1399로 상담하세요.  
[참고파일] audio/샘플_1.wav, audio/샘플_2.wav


In [55]:
print(chain.invoke("내일은 비가올까?"))

내일 비 예보에 대한 내용은 오디오에서 찾을 수 없습니다. [참고파일 없음]
